## TB granuloma evolution - extended model
    Marta Llabrés Tomás-Valiente

### Comments
- This baseline model aims to reproduce the spatio-temporal dynamics of a single TB granuloma. 
- This model is based on:
    - **Lesion**: evolution through a diffusion-reaction equation
    - **Fibroblasts**: evolution through both a chemokinetic and chemotactic term, and a proliferation term. The first one is proportional to the gradient of the lesion and is what drives the speed of the fibroblasts. The second one drives the direction of the movement (in principle) to tighten the encapsulation ring. The third one is to model the action of the TGF-$\beta$, and it is proportional to the gradient of the lesion.
    - **Calcification**: the necrosis of the lesion starts after encapsulation is completed, and no more oxygen nor nutrients can access within. The idea is that necrosis starts at the core of the lesion and expands towards the outsides. We have modelled it with an initial seed at the centre and then a reaction-diffusion equation. Keep in mind that the diffusion term does not model a *real* diffusion, but mimicks the growth of necrosis.

  The PDE system looks like:

  $$
\partial_t c_l = k_l g c_l (c_l-a) + \nabla \left( D_l g \nabla c_l \right) - k_{cal} c_{cal} c_l
$$

$$
\partial_t c_f = k_f g |\nabla c_l| c_f + \nabla \left ( \chi_{kin} g |\nabla c_l| \nabla c_f \right ) - \nabla \left ( \chi_{tax} g  c_f \nabla c_l \right )
$$

$$
\partial_t c_{cal} = k_{cal} c_{cal} c_l + \nabla \left( D_{cal}  \nabla c_{cal} \right)
$$


-  This **extended** version includes the possibility of lesion proliferation, and the new calcification dynamics

#### Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v2 as imageio
import os
import cv2
import csv
from scipy.stats import qmc
from shapely.geometry import Polygon, Point, box
from scipy.spatial import Voronoi, voronoi_plot_2d
from shapely import wkt, affinity
from typing import List, Tuple, Dict
from scipy.optimize import minimize
import statistics as stat
import matplotlib.ticker as ticker
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import matplotlib as mpl
from scipy.optimize import minimize
import statistics as stat
from scipy.ndimage import distance_transform_edt
import random

# ------- let's import our evolution function ----
import model2 as mfun4 # this is the full model

#### Functions

In [ ]:
### GEOMETRIC FUNCTIONS ###
# ============ Voronoi regions and space =================
def get_region_for_point(point, points, vor, lx, ly):
    idx = np.where((points == point).all(axis=1))[0][0]
    region_index = vor.point_region[idx]
    region = vor.regions[region_index]
    if -1 in region or len(region) == 0:
        return None  # Infinite region
    polygon = Polygon([vor.vertices[i] for i in region])
    return polygon.intersection(box(0, 0, lx, ly))

def random_point_in_polygon(polygon):
    minx, miny, maxx, maxy = polygon.bounds
    while True:
        # Sample uniformly in bounding box
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if polygon.contains(p):
            return p

In [ ]:
def calcul_radius_cells(m, dx):  
    return np.sqrt((dx**2)* np.sum(m > 0.05)/ (3.14)) #prenem l'area d'una esfera (en mm)

def calcul_radius(m, dx):  
    return np.sqrt((dx**2)* np.sum(m[m > 0.05])/ (3.14)) #prenem l'area d'una esfera (en mm)

#### Parameters

In [ ]:
# ======= FIXED PARAMETERS ========
    # space-time discretization
dx = 0.05  # [mm] (Size of an alveoli)
# amount of alveoli in a 2D secondary lobule of size 1cm2 = LxL with L=d*dx
lx = 150
ly = 180
dt = 0.00025 # [days]
iter = 200000

    #  for lesion initializa$ DCHemotion
a = 0.14 # threshold for lesion growth
eps = 1 # lesion sharpness
r0 = 1.5 # lesion initial radius in terms of grid cells
alfa = 1 # belongs to [0,1]. initial_concentration of lesion
count = 1 # initial number of lesions
age1 = 0 # initial age of lesion 1 (in days!!!)


# we create a dictionary with the fixed parameters
params_fixed = {"a": a, "eps": eps, "r0":r0, "alfa": alfa, "count": count, "dx": dx, "lx": lx, "ly": ly, "dt": dt, "age1": age1} 

# ======= ESTIMATED PARAMETERS ========
k = 9 # lesion reaction coefficient k_l
D_l = 5e-3 # lesion diffusion coefficient D_l
xi_kin = 0.9e-1 # coefficient of chemokinetic term for fibroblasts
xi_tax = 0.9e-1 # coefficient of chemotactic term for fibroblasts
k_f = 7.4 # fibroblasts reaction / proliferation coefficient
k_cal = 5 # calcification reaction coefficient
D_cal = 0.3e-2 # calcification "diffusion" coefficient
rho = 3e-5 #fitting parameter
alpha = 0.034638
n = 1.63483

k_nuc = 0.0035 
k_grow = 2.5
dcal = 0.001


param_names = ["k", "D", "xi", "xi_tax", "k_f", "k_nuc", "k_grow", "dcal", "rho", "alpha", "n"] 
best_params = [k, D_l, xi_kin, xi_tax, k_f, k_nuc, k_grow, dcal, rho, alpha, n]

#### Geometric initialization

In [ ]:
# ========= Voronoi tessellation ==========
pts = 2         
marge = 3     
centre = np.minimum(lx, ly)*3/4  

# Origin of lesion
i0 = np.random.randint(lx//2 - centre//2, lx//2 + centre//2)
j0 = np.random.randint(ly//2 - centre//2, ly//2 + centre//2)
central = np.array([i0, j0])

# Exterior points
exteriors = []
exteriors += [[np.random.randint(0, lx), np.random.randint(0, marge)] for _ in range(pts)]
exteriors += [[np.random.randint(0, marge), np.random.randint(0, ly)] for _ in range(pts)]
exteriors += [[np.random.randint(lx - marge, lx), np.random.randint(0, ly)] for _ in range(pts)]
exteriors += [[np.random.randint(0, lx), np.random.randint(ly - marge, ly)] for _ in range(pts)]
exteriors = np.array(exteriors)

all_points = np.vstack([central, exteriors])
vor = Voronoi(all_points)

central_region = get_region_for_point(central, all_points, vor, lx, ly)

params_fixed["origin1"] = central
params_fixed["central_region"] = central_region

# plot of first region
fig, ax = plt.subplots(figsize=(4, 4))
voronoi_plot_2d(
    vor,
    ax=ax,
    show_points=False,
    show_vertices=False,
    line_colors='black',
    line_width=0.75,
    line_alpha=1
)

if central_region is not None and not central_region.is_empty:
    x, y = central_region.exterior.xy
    ax.fill(x, y, color='darkgray', alpha=0.5, label='Secondary lobule')

ax.set_xlim(0, lx)
ax.set_ylim(0, ly)
ax.set_aspect('equal')

# Convert tick labels from grid cells to mm
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda val, _: f'{val*dx:.1f}'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, _: f'{val*dx:.1f}'))

ax.set_xlabel(r'$x$ (mm)')
ax.set_ylabel(r'$y$ (mm)')
ax.legend()
plt.tight_layout()
#plt.savefig("voronoi_area.pdf", dpi=300, bbox_inches='tight')
plt.show()

#### Execution of program

In [ ]:
# let's join the parameters
params = dict(zip(param_names, best_params))
params.update(params_fixed)
print(params.keys())

In [ ]:
# let's execute the full program
state, ages, origins, radius, radius_lc, full_radius, calcification, c_f1, max_grad, DIF_kin_vector, DIF_tax_vector, prol_vector, cal_dif_vec, probs, phases, i_encaps, encapsulation, roots = mfun4.model_evolution_loop(iter, params)

#### Results and plots

In [ ]:
x, y = central_region.exterior.xy
x_poly = np.array(x) * dx
y_poly = np.array(y) * dx

# heatmaps for the variables
cmap_cl = plt.cm.YlOrRd      # yellow to red, good for lesion
cmap_cf = plt.cm.YlGnBu      # yellow to blue, good for fibroblasts  
#cmap_ccal = plt.cm.Greys     # white to black, good for calcification
cmap_ccal = LinearSegmentedColormap.from_list('ccal', ['#ffffe5', '#737373', '#000000'])


# latex style configuration
mpl.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern Roman"], "axes.labelsize": 14, "font.size": 20, "legend.fontsize": 20, "xtick.labelsize": 12, "ytick.labelsize": 12})

# phases visualization
# --- colour map for each phase ---
phase_colors = {0:'#ffffff', 1:'#fff3cd', 2:'#ffd6a5', 3:'#cce5ff', 4:'#d4edda'}
phase_labels = {0: 'Disappeared', 1: 'Phase I: Growing', 2: 'Phase II: Encapsulating', 3: 'Phase III: Calcifying', 4: 'Phase IV: Resolved'}


In [ ]:
# data organization
c_l = state["c_l_1"].T 
c_f = c_f1.T
c_cal = calcification["c_cal_1"].T

total_c_l = sum(state.values())
total_c_cal = sum(calcification.values())

    # radius evolution variables
c_tot = c_l + c_f + c_cal
rad_les = radius["r_1"]
rad_calc = radius_lc["r_lc_1"]
full_radius = full_radius["full_r_1"]

time = np.arange(iter) * dt
extent_full = [0, c_l.shape[1]*dx, 0, c_l.shape[0]*dx]

In [ ]:
# radius evolution plot + background phases
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) lesion density
ax_cl = axes[0]
im_cl = ax_cl.imshow(total_c_l.T, cmap=cmap_cl, origin='lower', extent=extent_full, interpolation='bilinear', vmin = 0, vmax = 1)
ax_cl.plot(x_poly, y_poly, color='black', linewidth=.75, label='Lobule Boundary')
ax_cl.set_xlabel(r'$x$ (mm)', size = 17)
ax_cl.set_ylabel(r'$y$ (mm)', size = 17)
#ax_cl.set_title(r'(a)', loc='left')
divider = make_axes_locatable(ax_cl)
cax_cl = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cl, cax=cax_cl, label=r'$c_l$')

# 2) fibroblast density
ax_cf = axes[1]
im_cf = ax_cf.imshow(c_f1.T, cmap=cmap_cf, origin='lower', extent=extent_full, interpolation='bilinear', vmin = 0, vmax = 1)
ax_cf.plot(x_poly, y_poly, color='black', linewidth=.75, label='Lobule Boundary')
ax_cf.set_xlabel(r'$x$ (mm)', size = 17)
ax_cf.set_ylabel(r'$y$ (mm)', size = 17)
#ax_cf.set_title(r'(b)', loc='left')
divider2 = make_axes_locatable(ax_cf)
cax_cf = divider2.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cf, cax=cax_cf, label=r'$c_f$')

# 3) calcification density
ax_ccal = axes[2]
cal_max = np.max(total_c_cal) if np.max(total_c_cal) > 1e-6 else 1.0  # fallback if empty
im_ccal = ax_ccal.imshow(total_c_cal.T, cmap=cmap_ccal, origin='lower', extent=extent_full, interpolation='bilinear', vmin=0, vmax=1)
ax_ccal.plot(x_poly, y_poly, color='black', linewidth=.75, label='Lobule Boundary')
ax_ccal.set_xlabel(r'$x$ (mm)', size = 17)
ax_ccal.set_ylabel(r'$y$ (mm)', size = 17)
#ax_ccal.set_title(r'(c)', loc='left')
divider3 = make_axes_locatable(ax_ccal)
cax_ccal = divider3.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_ccal, cax=cax_ccal, label=r'$c_{cal}$')

plt.tight_layout()
#plt.savefig("model_vor.pdf", dpi=300, bbox_inches='tight')
plt.show()